# LeNet 1 Paper (1989)

## Objective
1. To a gain a better understanding of LeNet image classification model introdued in the paper **[Backpropagation Applied to Handwritten Zip Code Recognition](https://galileo-unbound.blog/wp-content/uploads/2025/02/lecun.neco_.1989.1.4.541.pdf)**.


**Note**:
- To look at the entire model's implementation at one place you can have a look at the model's code written here `computer-vision/src/computer_vision/classification/lenet_1.py`
- To understand convolution neural network layer in depth you can look at [Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb) (** I advised you to go through this notebook before you go ahead for better understanding of convolution layers**)

## Introduction

The authors of the paper emphasize the use of convolution function in the neural network world to perform vision related tasks. The layer that uses convolution is popularly known as **Convolution Layer**. It has been observed that when neural network layers are built to extract low level features from original input data and combined to build high level feature the models are able to train better to perform the task at hand. Prior to this paper this was done through linear neural work layers and to propagate input through these layers the input features need to bed flattened out in a single linear order.

These linear neural network layers have their own disadvantages:
1. Due to flattening of original input feature we lose spatial or temporal inforamtion stored within the structure of the input its raw form. To work with linear neural network layers and to ensure the spatial or temporal information is not lost heavy **feature engineering** needs to be done in order to retain the information as part of the input.
2. The linear neural network layers are dense fully connected layers. The total number of trainable weights that represent a single layer is equal to **$InputDimension * OutputDimension + Bias$**. The total number of dimensions grow exponentially as the number of layers increases.

The authors did not want people to spend a large amount of time feature engineering instead wanted these features to be learned as part of the model training and did not want to use large number of weights and biases to learn these features for this they preferred convolution neural networks.

LeNet introduces certain principles and a model template that is still followed by latest models as well. It is very important for people to understand this paper properly.

Before we dive into the details of LeNet I want you to understand what is **classification** and how is the approach taken in **image classification** different from other types of tasks.

## What is a standard classification task?

The object of machine learning and artificial intelligence is to find patterns, learn them and predict an outcome for a predefined task based on learnt patterns. A standard classification is a predictive task where patterns are learned from a data's features/attributes and they are categorised/labeled into a particular category. These categories are defined during the process of defining the problem statement for which classification is done.

Classification works with the assumption that the classes are **mutually exclusive** in nature i.e. a data can only belong to only one class at a time. If such data exists where a data can belong to more than one class then either the problem statement should shift from a standard classification problem to a **multi-label** classification problem or the data does not have sufficient features or attributes to allow a model to establish a clear differentiation boundary within the classes.

There are many types of classification problems which are governed by factors such as modal of data, single or multi label classification. Here we are interested in classifying images into different categories hence our focus is on **image classification**

### What is an image?
Images are objects that store visual information in a digital electronic form. Images are commonly stored as **gray-scale images** or **RGB images**. The smalled unit that stores information in an image is called a **pixel**. This pixel stores the intensity of light and this value has a range of [0, 225].
- 0 means complete lack of light
- 255 means maximum light intensity is present in this pixel

An image's boundary is defined by it's **width** and **height**. In compute vision a source of information is called a **channel**.

#### 1. Gray-scale Images
- A gray-scale image is an image which only store the intensity of white light and hence its a black and white image. 
- A gray-scale image contains information about only white light and hence it is called a **single channel** image. 
- A pixel in a gray-scale image will only store 1 value between 0 and 255

#### 2. RGB Images
- A RGB image contains information about the intensity of **red**, **green** and **blue** light.
- A RGB image is called a **three channel** image.
- A pixel in a RGB image will store 3 values one for each light.

The images are identified with the shape $(Width,Height,Channel)$

Images inherently store spatial information which encompasses distinct patterns observed visually and their relative position with respect to each other. When we train our computer vision models our priority is to learn the local structure (**distinct features**) and relative spatial geometry (**relative position of features with respect to each other**) over absolute position of the features within the image. Let's understand this with an example.

In an image of a human body a model should not explicitly map the feature representing the "head" to the upper boundaries of the image nor the feature representing the "feet" to the lower boundaries. Instead it must learn the relative position between them which is the head structurally exists at the opposite vertical end relative to the feet. If the image undergoes a spatial transformations such as **$180^o$** rotation a robust model should have the classification capacity by relying on these relative feature configurations rather than failing due to the shift in absolute spatial coordinates.

### What is image classification?
Image classification is a task that trains a machine learning model on visual data (images) to map the local structure (distinct features) and their relative spatial geometry to distinct classes. Keep this definition in mind while going through the next section on design philosophy as the template it introduces follows the above definition and is used by almost all of the image classification models

**Note**<br>
In machine learning world to build computer vision models the standard shape of images we work with is **$(Channel,Height,Width)$**.The values stored in the pixel need to be transformed and scaled from a range of **$[0-255]$** to **$[0-1]$**. So before training your models make sure a transformation is applied on the images to:<br>
1. Convert their shape from **$(Width,Height,Channel)$** to **$(Channel,Height,Width)$**.
2. Rescale the range of values from **$[0-255]$** to **$[0-1]$**


## Design philosophy

The authors of the paper wanted to achieve the following goals:<br>
1. To learn high level information from low level features provided as images
2. To not use linear neural network layers to learn high level information features
3. To retain the local construct (local distinct features) information present in images
4. To learn relative spatial geometry (relative positioning of features with respect to each other)
5. To discard absolute positioning of features in the images

Before we understand how the authors were able to achieve their goal lets understand few key terms as these are used through out computer vision papers and will help you understand their design philosophy a little better.<br>
- **weights**, **bias**, **convolution filter**, **input tensor**, **output tensor**, **kernel size**, **stride**, **padding** and **dilation** have been explained in **[Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb)**
- **Feature Map**: This is the result of the convolution function between the input tensor and the convolution filter. A **feature map** represents one channel of the output tensor of dimension **$(height, width)$**. 
- **Under Sampling**/**Down Sampling**/**Pooling**: The process of reducing the number of features propagated to the next layer as part of the output tensor. This is done in order to discard the information related to absolute position of the features in the image

Now we will look at how the authors achieved their goals
### 1. Learning high level features from low level features (images) without using linear layers (covers goal 1 and 2)
The authors achieved their goal of **learning high level features from low level features (images) without using linear layers** by using convolution functions (convolution filters). linear layers require a new set of weights and biases between the entire input layer and every neuron in the output layer. This exponentially increases the number of trainable parameters of the model. Convolution filters elimintate the large number weights and biases needed by introducing **sharable weights and biases** for each channel in the input tensor. Lets understand how this eliminates the presence of large number of parameters with an example.

Example: We have an input tensor of $N$ channels where each channel has information stored related to input tensor as a 2 dimensional matrix of shape $(H_{in}, $W_{in})$ and we generate an output of $M$ channels of 2 dimensional matrices of shape $(H_{out},W_{out})$. (Use $N$ = 3, $H_{in}$ = 224, $W_{in}$ = 224, $M$=6, $H_{out}$ = 112, $W_{out}$ = 112) (2 fold the number of channels and half the size of feature maps in a single step)

#### Case 1: Using linear layers
A linear layer will always required a flattened input and will always generate a flattened output. So first we need to flatten the input and calculate the number of elements in it. <br>
**Number of elements in the input** = $N*H_{in}*W_{in} = 3 * 224 * 224 = 150,528$<br>
**Number of elements in the output** = $M*H_{out}*W_{out} = 6 * 112 * 112 = 75,264$<br>

We know that every element in the output is connected to every element in the input and hence we would need **$N*H_{in}*W_{in}*M*H_{out}*W_{out}$** weights and **$M*H_{out}*W_{out}$** biases, which will be equal to $(150,528 * 75,264) + 75,264 = 11,329,414,656$ training parameters **~11 billion 3 hundred and 30 million parameters!** 


#### Case 2: Using convolution filters (layers)
When using convolution filters we do not need to flatten the input. We need $M$ sets of $(N,KernelSize, KernelSize)$ weights + $M$ biases. The only unknowns we have is $KernelSize$. We can find it's value by using the following formula
$$H_{out} = \lfloor(\frac{H_{in} + (2 * padding) - (dilation * (kernelsize - 1) - 1)}{stride})\rfloor - 1$$
$$W_{out} = \lfloor(\frac{W_{in} + (2 * padding) - (dilation * (kernelsize - 1) - 1)}{stride})\rfloor - 1$$
where:<br>
$H_{out}$ = 112, $W_{out}$ = 112, $H_{in}$ = 224, $W_{in}$ = 224, $dilation$ = 1, $stride$ = 2 (since we want to half the features), $padding$ = 1<br>
$kernelsize$ comes out to be $(3,3)$<br>

Therefore we would need **6** sets of $(3,3,3)$ weights and **6** biases. The total comes out to be $(6*3*3*3) + 6 = 168$. If we use convolution filters we only need **168** parameters which is almost nothing compared to what we saw in the case of linear neural network layers.

To learn features which store information related to correlations between lower level features we would need multiple steps/layers of neural networks to reach these high level features. In the case of linear neural network layers this seems to hold true, but when it comes to convolution neural network layers to create more number of high level features we have three approaches.<br>
  1. Increae the number of channels as a result of convolution in the next layer
  2. Increase the number of layers of convolution.
  3. Increase both the number of channels and number of layers of convolution

In a convolution neural network layer a channel represents a source of information having more channels allows the network to capture a richer variety of correlations from the incoming layers. This is one of the main principles which is part of the template followed by most of the computer vision models which is **as you increase the depth of the model you should also increase the number of channels in the convolutional layers.** This guarantees the model has enough capacity to store diverse high-level correlations derived from the lower-level inputs.

### 2. Learn local construct (covers goal 3 and 4)
Convolution layers share the same set of weights and bias as they silde across a channel they will always produce the same output for identical input patterns. The size of the kernel dictates the size of the receptive field on a channel of the input tensor. A small kernel (3x3) has a small receptive field that excels at capturing fine localized details like edges and as we start increasing the size of the filters the receptive field increases. Increase in the receptive field helps capture broader more generalized context it often comes at the expense of losing fine-grained spatial information. This advantage that convolution filter brings allows us to capture local construct and relative spatial geometry.

### 3. Discarding absolute positioning of features in an image (covers goal 5)
Allowing a model to learn the absolute position of features within an image is highly problematic. When this happens the network learns the feature's fixed location in the frame rather than its context within the object itself. For example, if a model expects a specific feature to only appear at the top of an image it will fail to classify a 180° vertically rotated version of that exact same image.

To prevent this we remove absolute positioning through a process called **pooling** (historically referred to as **subsampling** or **undersampling** in early architectures like LeNet). Pooling reduces the spatial dimensions of the feature map by aggregating the values within a receptive field typically by taking the **maximum** or **average**. Unlike convolutional layers pooling layers do not use learnable weights or biases. By summarizing the information from the previous layer. Pooling creates spatial invariance allowing the network to recognize a feature's presence while discarding its exact absolute coordinates.

## Design Template

Computer vision models generally follow a two-part structure:

1. **Extractor**:
   1. **Feature Extraction**: This phase builds complex high-level features from raw inputs using convolutional layers and their accessory layers.
   2. **Scaling Complexity**: To capture richer patterns, the network's depth (more layers), width (more feature maps per layer), or both are progressively increased.
   3. **Discarding Absolute Position**: Pooling (or subsampling) layers are used throughout the extraction phase to discard exact coordinate data, ensuring the model focuses on what the feature is rather than exactly where it is.

2. **Classifier**: The final phase takes the high-level features generated by the extraction layers and categorizes them into mutually exclusive classes.

Keeping this structure in mind will help you understand computer vision papers much more easily.

## LeNet Architecture and Training Pipeline

Yann LeCunn and other's developed this model in the year 1989. The model has been trained on USPS (United States Postal Service) dataset which contains gray scale images of size 16x16 pixels of handwritten digits.

### Dataset
The dataset comprises of 7,291 instances 16x16 gray scale images in the train set and 2,007 instances in the test set

### Model Architecture
The model comprises of 3 layers h1, h2 and h3. 2 layers are part of the **extractor** block **(h1, h2)** and 1 layer is part of the classifier block **(h3)**.

#### H1 Layer
Unlike modern layers where in a convolution layer 1 bias or no bias is alloted to each convolution filter. in H1 layer of LeNet instead of allocating 1 bias to each convolution fitler they have allocated 1 bias to each element of each feature map. The H1 layer takes in an input of (1 x 16 x 16) and returns an output of (12, 8, 8). To achieve the output the dimensions of the convolution filter are $kernelSize = 5, padding = 2, stride = 2, dilation = 1, bias = False$ and after the convolution operation biases of the shape (12, 8, 8) are added to the result. 

#### H2 Layer
H2 Layer generates 12 feature maps. Here instead of allowing all 12 feature maps (obtained from H1) to be connected to the 12 feature maps of H2 layer Yann LeCunn and others created a set of 12 sets of 8 input feature maps where each set is responsible for one feature map in the output. In the paper they have not disclosed the combination in which these input feature maps where clubbed together. This approach was taken by the authors because they wanted to force the model to learn different correlation information by each output feature map of the h2 layer

(For simplification I took reference from LeNet 5 and created my combination of 12 sets of 8 input feature maps) Below is the table in which all feature maps were combined in each set. **X** represents that feature map was part of the feature set. The columns represent the feature set number. The row represent the feature map number.

| Feature Set Number -> | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
| :--- | :---: | ---: | :--- | :---: | ---: | :--- | :---: | ---: |:--- | :---: | ---: | ---: |
| Feature Map 1 | X | | | | | X | X | X | X | X | X | X |
| Feature Map 2 | X | X | | | | | X | X | X | X | X | X |
| Feature Map 3 | X | X | X | | | | | X | X | X | X | X |
| Feature Map 4 | X | X | X | X | | | | | X | X | X | X |
| Feature Map 5 | X | X | X | X | X | | | | | X | X | X |
| Feature Map 6 | X | X | X | X | X | X | | | | | X | X |
| Feature Map 7 | X | X | X | X | X | X | X | | | | | X |
| Feature Map 8 | X | X | X | X | X | X | X | X | | | | |
| Feature Map 9 | | X | X | X | X | X | X | X | X | | | |
| Feature Map 10 | | | X | X | X | X | X | X | X | X | | |
| Feature Map 11 | | | | X | X | X | X | X | X | X | X | |
| Feature Map 12 | | | | | X | X | X | X | X | X | X | X |

All 12 feature set use the following values for the convolution filters<br>
$kernelSize = 5, padding = 2, stride = 2, dilation = 1, bias = False$.

During the convolution stage The output from H1 layer will be segregated into 12 groups based on the above table. After all 12 sets convolve we combine them (**concatenate**) on the channel dimension.

The output of H2 is (12, 4, 4) this output go through a flattening operation so that it can flow through the H3 layer. 

#### H3 Layer
H3 layer in the paper takes in (12 x 4 x 4  = 192 parameters) and creates an output of 30 parameters which goes through another layer to get 10 outputs

The total number of trainable parameters is 9760.

#### Non Linear Activation Function
Tanh was used as the non linear activation function between two layers as (Tanh allows for faster convergence as compared to sigmoid (at that time two prominant non linear activation function were Tanh and Sigmoid)).

**Note**: There is an experiment that i have done to compare the convergence rate of models when tanh and sigmoid are used. Here is a link to it [Vanishing Gradients Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/experiments/01_convergence_analysis_sigmoid_vs_tanh.ipynb)

### Training Pipeline
The model was trained on the dataset for 23 epochs and unlike cross entropy loss seen normally in case of multi class classification Mean Squared Error (MSE) Loss was used as the cost function

The entire model's implementation is here `computer-vision/src/computer_vision/classification/lenet_1.py`

Below is the model's summary.

In [1]:
from torchinfo import summary
from computer_vision.classification.lenet_1 import LeNet

summary(LeNet(), input_size=(1, 1, 16, 16), col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
LeNet                                    [1, 1, 16, 16]            [1, 10]                   --                        960
├─Conv2d: 1-1                            [1, 1, 16, 16]            [1, 12, 8, 8]             [5, 5]                    300
├─Tanh: 1-2                              [1, 12, 8, 8]             [1, 12, 8, 8]             --                        --
├─Conv2d: 1-3                            [1, 8, 8, 8]              [1, 1, 4, 4]              [5, 5]                    200
├─Conv2d: 1-4                            [1, 8, 8, 8]              [1, 1, 4, 4]              [5, 5]                    200
├─Conv2d: 1-5                            [1, 8, 8, 8]              [1, 1, 4, 4]              [5, 5]                    200
├─Conv2d: 1-6                            [1, 8, 8, 8]              [1, 1, 4, 4]              [5, 5]                    200
├─Conv2d: 1-7